In [0]:
import pyspark.sql.functions as F
import pyspark.databricks.sql.functions as DBF

In [0]:
catalog = "stuart"
schema = "lv"

h3_resolution = 8

In [0]:
raw_df = spark.table(f"{catalog}.{schema}.raw_raster")

In [0]:
indexed_df = (
  raw_df
  .withColumn("geom_UTM", DBF.st_point("x", "y", "srid"))
  .withColumn("geom_4326", DBF.st_transform("geom_UTM", 4326))
  .withColumn(f"h3_r{h3_resolution}", DBF.h3_pointash3(DBF.st_asbinary("geom_4326"), h3_resolution))
)

display(indexed_df)

In [0]:
result_df = (
    indexed_df
    .groupBy("sensing_time", "variable", "h3_r8")
    .agg(
        F.count("m").alias("count_m"),
        F.mean("m").alias("mean_m"),
        F.percentile("m", 0.25).alias("lower_quartile_m"),
        F.percentile("m", 0.50).alias("median_m"),
        F.percentile("m", 0.75).alias("upper_quartile_m"),
        F.min("m").alias("min_m"),
        F.max("m").alias("max_m")
    )
)

display(result_df)

In [0]:
(
  result_df.write
  .mode("overwrite")
  .saveAsTable(f"{catalog}.{schema}.result_raster")
  )

In [0]:
spark.sql(f"""
ALTER TABLE {catalog}.{schema}.result_raster
CLUSTER BY (variable, h3_r8, sensing_time)
          """)